In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os, json, glob, time, copy
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.metrics import f1_score, confusion_matrix, classification_report

device = 'cuda'
IN = os.path.dirname(glob.glob('/kaggle/input/**/feats_b0.npy', recursive=True)[0])
meta = pd.read_csv(f'{IN}/metadata.csv')
cfg = json.load(open(f'{IN}/config.json'))
classes = cfg['classes']; C = len(classes)

# Whole dataset fits on the GPU (~800 MB in float16), so training is very fast
X  = torch.from_numpy(np.load(f'{IN}/feats_b0.npy')).to(device)
Xf = torch.from_numpy(np.load(f'{IN}/feats_b0_flip.npy')).to(device)
y  = torch.tensor(meta['label'].values, device=device)

def split_idx(col, s):
    return torch.tensor(np.where(meta[col] == s)[0], device=device)

print('Features:', tuple(X.shape), '| classes:', C)
print(meta['split_match'].value_counts())

Features: (10091, 16, 1280) | classes: 15
split_match
train    7254
val      1487
test     1350
Name: count, dtype: int64


In [2]:
class TemporalAttention(nn.Module):
    """Gives each frame a score, softmax over time, weighted sum of frame outputs."""
    def __init__(self, d, h=128):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(d, h), nn.Tanh(), nn.Linear(h, 1))
    def forward(self, H):                                   # H: (B, T, D)
        a = torch.softmax(self.score(H).squeeze(-1), dim=1) # (B, T), sums to 1 over frames
        return (a.unsqueeze(-1) * H).sum(1), a              # context (B, D), weights (B, T)

class ShotClassifier(nn.Module):
    def __init__(self, kind='gru', bidirectional=False, attention=False,
                 in_dim=1280, hidden=256, layers=2, num_classes=15, dropout=0.3):
        super().__init__()
        self.kind, self.bi, self.use_attn = kind, bidirectional, attention
        self.norm = nn.LayerNorm(in_dim)
        self.in_drop = nn.Dropout(0.2)
        if kind == 'mean':                       # baseline: no temporal modelling at all
            out_dim = in_dim
        else:
            rnn_cls = {'rnn': nn.RNN, 'lstm': nn.LSTM, 'gru': nn.GRU}[kind]
            self.rnn = rnn_cls(in_dim, hidden, num_layers=layers, batch_first=True,
                               dropout=dropout if layers > 1 else 0, bidirectional=bidirectional)
            out_dim = hidden * (2 if bidirectional else 1)
            if attention:
                self.attn = TemporalAttention(out_dim)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(out_dim, 256), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(256, num_classes))

    def forward(self, x):                        # x: (B, T=16, 1280)
        x = self.in_drop(self.norm(x))
        attn = None
        if self.kind == 'mean':
            z = x.mean(1)                        # average over frames, so frame order is ignored
        else:
            out, h = self.rnn(x)                 # out: (B, T, D), h: final hidden states
            if self.use_attn:
                z, attn = self.attn(out)
            else:
                if self.kind == 'lstm':
                    h = h[0]                     # LSTM returns (h, c); use h
                z = torch.cat([h[-2], h[-1]], 1) if self.bi else h[-1]  # last layer's final state
        return self.head(z), attn

m = ShotClassifier('gru', True, True).to(device)
print('Bi-GRU+attn params:', sum(p.numel() for p in m.parameters()) / 1e6, 'M')
print(m(torch.randn(2, 16, 1280, device=device))[0].shape)   # expect (2, 15)

Bi-GRU+attn params: 3.748624 M
torch.Size([2, 15])


In [3]:
@torch.no_grad()
def predict(model, ids, bs=512):
    model.eval()
    return torch.cat([model(X[ids[i:i+bs]].float())[0] for i in range(0, len(ids), bs)])

def evaluate(model, ids):
    logits = predict(model, ids); yt = y[ids]; pred = logits.argmax(1)
    return {'acc': (pred == yt).float().mean().item(),
            'top3': (logits.topk(3, 1).indices == yt[:, None]).any(1).float().mean().item(),
            'macro_f1': f1_score(yt.cpu(), pred.cpu(), average='macro')}

def run(config, split_col='split_match', seed=0, epochs=40, bs=64, lr=1e-3,
        patience=8, use_flip=True, verbose=False):
    torch.manual_seed(seed); np.random.seed(seed)
    tr, va, te = [split_idx(split_col, s) for s in ['train', 'val', 'test']]

    counts = torch.bincount(y[tr], minlength=C).float()
    crit = nn.CrossEntropyLoss(weight=counts.sum() / (C * counts), label_smoothing=0.1)  # class weights for imbalance

    model = ShotClassifier(**config, num_classes=C).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=3)

    best_f1, best_state, best_val, bad = -1, None, None, 0
    for ep in range(1, epochs + 1):
        model.train()
        perm = tr[torch.randperm(len(tr), device=device)]
        for i in range(0, len(perm), bs):
            b = perm[i:i+bs]
            xb = X[b].float()
            if use_flip:                                   # flip augmentation: training set only
                mask = torch.rand(len(b), device=device) < 0.5
                xb[mask] = Xf[b[mask]].float()
            loss = crit(model(xb)[0], y[b])
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # prevents exploding gradients in RNNs
            opt.step()
        val = evaluate(model, va)
        sched.step(val['macro_f1'])
        if verbose:
            print(f"ep {ep:2d} loss {loss.item():.3f} val acc {val['acc']:.3f} f1 {val['macro_f1']:.3f}")
        if val['macro_f1'] > best_f1:                      # early stopping on VALIDATION, never test
            best_f1, best_val, best_state, bad = val['macro_f1'], val, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_val, evaluate(model, te), ep

# Quick sanity check: one GRU+attention run
t0 = time.time()
model, val, test, ep = run(dict(kind='gru', attention=True), verbose=True)
print(f'\nval {val}\ntest {test}\nepochs {ep}, {time.time() - t0:.0f}s')

ep  1 loss 2.669 val acc 0.237 f1 0.180
ep  2 loss 2.068 val acc 0.418 f1 0.409
ep  3 loss 1.794 val acc 0.484 f1 0.461
ep  4 loss 1.575 val acc 0.511 f1 0.501
ep  5 loss 1.787 val acc 0.551 f1 0.535
ep  6 loss 1.857 val acc 0.582 f1 0.573
ep  7 loss 1.988 val acc 0.553 f1 0.542
ep  8 loss 1.068 val acc 0.573 f1 0.565
ep  9 loss 1.694 val acc 0.576 f1 0.568
ep 10 loss 1.502 val acc 0.588 f1 0.581
ep 11 loss 1.758 val acc 0.575 f1 0.573
ep 12 loss 1.042 val acc 0.608 f1 0.599
ep 13 loss 1.189 val acc 0.611 f1 0.599
ep 14 loss 1.191 val acc 0.594 f1 0.588
ep 15 loss 0.813 val acc 0.610 f1 0.607
ep 16 loss 1.068 val acc 0.610 f1 0.607
ep 17 loss 1.127 val acc 0.608 f1 0.596
ep 18 loss 1.099 val acc 0.605 f1 0.602
ep 19 loss 0.859 val acc 0.606 f1 0.594
ep 20 loss 0.932 val acc 0.622 f1 0.608
ep 21 loss 0.947 val acc 0.618 f1 0.613
ep 22 loss 0.838 val acc 0.618 f1 0.609
ep 23 loss 0.869 val acc 0.613 f1 0.614
ep 24 loss 0.718 val acc 0.614 f1 0.609
ep 25 loss 0.694 val acc 0.615 f1 0.613


In [8]:
CONFIGS = {
    'Mean-pool (no RNN)': dict(kind='mean'),
    'RNN':                dict(kind='rnn'),
    'LSTM':               dict(kind='lstm'),
    'GRU':                dict(kind='gru'),
    'Bi-GRU':             dict(kind='gru', bidirectional=True),
    'RNN + attn':         dict(kind='rnn', attention=True),
    'LSTM + attn':        dict(kind='lstm', attention=True),
    'GRU + attn':         dict(kind='gru', attention=True),
    'Bi-GRU + attn':      dict(kind='gru', bidirectional=True, attention=True),
}
SEEDS = [0, 1, 2]
results, models = [], {}
for name, c in CONFIGS.items():
    for s in SEEDS:
        t0 = time.time()
        model, val, test, ep = run(c, seed=s)
        params = sum(p.numel() for p in model.parameters())
        results.append({'model': name, 'seed': s, 'epochs': ep, 'params_M': params / 1e6,
                        'val_f1': val['macro_f1'], 'test_acc': test['acc'],
                        'test_top3': test['top3'], 'test_f1': test['macro_f1']})
        if s == 0:
            models[name] = model
        print(f"{name:20s} seed {s}: test acc {test['acc']:.3f} f1 {test['macro_f1']:.3f} ({time.time()-t0:.0f}s)")

res = pd.DataFrame(results)
res.to_csv('/kaggle/working/results_all_runs.csv', index=False)
summary = res.groupby('model', sort=False).agg(
    params_M=('params_M', 'first'),
    val_f1=('val_f1', 'mean'),
    test_acc_mean=('test_acc', 'mean'), test_acc_std=('test_acc', 'std'),
    test_top3=('test_top3', 'mean'),
    test_f1_mean=('test_f1', 'mean'), test_f1_std=('test_f1', 'std')).round(4)
summary.to_csv('/kaggle/working/results_summary.csv')
summary

In [ ]:
best_name = summary['val_f1'].idxmax()          # choose the best model by VALIDATION, not test
print('Best model (by val F1):', best_name)

leak = []
for s in SEEDS:
    _, _, t_rand, _ = run(CONFIGS[best_name], split_col='split_random', seed=s)
    leak.append({'split': 'random (leaky)', 'seed': s, 'test_acc': t_rand['acc'], 'test_f1': t_rand['macro_f1']})
for _, r in res[res.model == best_name].iterrows():
    leak.append({'split': 'by match (fair)', 'seed': r.seed, 'test_acc': r.test_acc, 'test_f1': r.test_f1})
leak = pd.DataFrame(leak)
leak.to_csv('/kaggle/working/results_leakage.csv', index=False)
leak.groupby('split')[['test_acc', 'test_f1']].agg(['mean', 'std']).round(4)

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns

te = split_idx('split_match', 'test')
best = models[best_name]
pred = predict(best, te).argmax(1).cpu().numpy(); yt = y[te].cpu().numpy()

print(classification_report(yt, pred, target_names=classes, digits=3))
cm = confusion_matrix(yt, pred, normalize='true')
plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(f'{best_name}: test confusion matrix (row-normalised)')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150); plt.show()

In [ ]:
attn_name = 'Bi-GRU + attn'
am = models[attn_name].eval()
ids = te[:6]
with torch.no_grad():
    logits, a = am(X[ids].float())
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for ax, i, w, p in zip(axes.flat, ids.cpu().numpy(), a.cpu().numpy(), logits.argmax(1).cpu().numpy()):
    ax.bar(range(16), w)
    ax.set_title(f"true: {classes[meta.label[i]]}\npred: {classes[p]}", fontsize=9)
    ax.set_xlabel('frame'); ax.set_ylim(0, w.max() * 1.2)
plt.suptitle('Temporal attention weights'); plt.tight_layout()
plt.savefig('/kaggle/working/attention_examples.png', dpi=150); plt.show()

In [ ]:
torch.save({'state_dict': models[best_name].state_dict(), 'config': CONFIGS[best_name],
            'classes': classes, 'num_frames': cfg['num_frames']}, '/kaggle/working/best_model.pt')
torch.save({'state_dict': models['Bi-GRU + attn'].state_dict(), 'config': CONFIGS['Bi-GRU + attn'],
            'classes': classes, 'num_frames': cfg['num_frames']}, '/kaggle/working/bigru_attn_model.pt')
!ls -lh /kaggle/working

In [ ]:
CONFIGS = {
    'Mean-pool (no RNN)': dict(kind='mean'),
    'RNN':                dict(kind='rnn'),
    'LSTM':               dict(kind='lstm'),
    'GRU':                dict(kind='gru'),
    'Bi-GRU':             dict(kind='gru', bidirectional=True),
    'RNN + attn':         dict(kind='rnn', attention=True),
    'LSTM + attn':        dict(kind='lstm', attention=True),
    'GRU + attn':         dict(kind='gru', attention=True),
    'Bi-GRU + attn':      dict(kind='gru', bidirectional=True, attention=True),
}
SEEDS = [0, 1, 2]

In [10]:
IN_CROP = os.path.dirname(glob.glob('/kaggle/input/**/feats_b0_crop_frame.npy', recursive=True)[0])
meta_crop = pd.read_csv(f'{IN_CROP}/metadata.csv')
assert (meta_crop.path.values == meta.path.values).all(), 'Clip order differs between notebooks!'

FEATS = {
    'Full frame':     (f'{IN}/feats_b0.npy',                 f'{IN}/feats_b0_flip.npy'),
    'Per-frame crop': (f'{IN_CROP}/feats_b0_crop_frame.npy', f'{IN_CROP}/feats_b0_crop_frame_flip.npy'),
    'Per-clip crop':  (f'{IN_CROP}/feats_b0_crop_clip.npy',  f'{IN_CROP}/feats_b0_crop_clip_flip.npy'),
}
print('Same clips and order in both notebooks ✅')

Same clips and order in both notebooks ✅


In [11]:
compare_models = ['Mean-pool (no RNN)', 'LSTM + attn', 'GRU + attn', 'Bi-GRU + attn']
rows, crop_models = [], {}
for feat_name, (fa, fb) in FEATS.items():
    X  = torch.from_numpy(np.load(fa)).to(device)      # run() and predict() use these global X / Xf
    Xf = torch.from_numpy(np.load(fb)).to(device)
    for name in compare_models:
        for s in SEEDS:
            model, val, test, ep = run(CONFIGS[name], seed=s)
            rows.append({'input': feat_name, 'model': name, 'seed': s, 'val_f1': val['macro_f1'],
                         'test_acc': test['acc'], 'test_top3': test['top3'], 'test_f1': test['macro_f1']})
            if s == 0:
                crop_models[(feat_name, name)] = model
            print(f"{feat_name:15s} {name:20s} seed {s}: acc {test['acc']:.3f} f1 {test['macro_f1']:.3f}")

crop_res = pd.DataFrame(rows)
crop_res.to_csv('/kaggle/working/results_crop_comparison.csv', index=False)
crop_summary = crop_res.groupby(['model', 'input'], sort=False).agg(
    val_f1=('val_f1', 'mean'), test_acc=('test_acc', 'mean'), acc_std=('test_acc', 'std'),
    test_top3=('test_top3', 'mean'), test_f1=('test_f1', 'mean')).round(4)
crop_summary.to_csv('/kaggle/working/results_crop_summary.csv')
crop_summary

Full frame      Mean-pool (no RNN)   seed 0: acc 0.447 f1 0.443
Full frame      Mean-pool (no RNN)   seed 1: acc 0.456 f1 0.447
Full frame      Mean-pool (no RNN)   seed 2: acc 0.456 f1 0.450
Full frame      LSTM + attn          seed 0: acc 0.608 f1 0.600
Full frame      LSTM + attn          seed 1: acc 0.630 f1 0.623
Full frame      LSTM + attn          seed 2: acc 0.634 f1 0.631
Full frame      GRU + attn           seed 0: acc 0.647 f1 0.621
Full frame      GRU + attn           seed 1: acc 0.621 f1 0.618
Full frame      GRU + attn           seed 2: acc 0.616 f1 0.604
Full frame      Bi-GRU + attn        seed 0: acc 0.642 f1 0.629
Full frame      Bi-GRU + attn        seed 1: acc 0.647 f1 0.632
Full frame      Bi-GRU + attn        seed 2: acc 0.644 f1 0.623
Per-frame crop  Mean-pool (no RNN)   seed 0: acc 0.539 f1 0.526
Per-frame crop  Mean-pool (no RNN)   seed 1: acc 0.553 f1 0.541
Per-frame crop  Mean-pool (no RNN)   seed 2: acc 0.558 f1 0.553
Per-frame crop  LSTM + attn          see

,,val_f1,test_acc,acc_std,test_top3,test_f1
model,input,,,,,
Mean-pool (no RNN),Full frame,0.4467,0.4531,0.0049,0.7622,0.4466
LSTM + attn,Full frame,0.6068,0.6242,0.0140,0.8709,0.6180
GRU + attn,Full frame,0.6227,0.6281,0.0168,0.8286,0.6146
Bi-GRU + attn,Full frame,0.6301,0.6444,0.0027,0.8519,0.6283
Mean-pool (no RNN),Per-frame crop,0.5504,0.5501,0.0097,0.8291,0.5402
LSTM + attn,Per-frame crop,0.6916,0.6894,0.0089,0.8667,0.6851
GRU + attn,Per-frame crop,0.6902,0.6958,0.0083,0.8548,0.6907
Bi-GRU + attn,Per-frame crop,0.7032,0.7195,0.0070,0.8741,0.7118
Mean-pool (no RNN),Per-clip crop,0.5275,0.5200,0.0109,0.8111,0.5089
